In [1]:
import os, pandas as pd, shutil; from pathlib import Path; from glob import glob

In [2]:
nifty_pdf_data_path = Path("D:\\NIFTY Stock backtest\\NSE pdf data")
folders = os.listdir(nifty_pdf_data_path)

In [5]:
output_dir = Path("pdf data")
output_dir.mkdir(parents=True, exist_ok=True)

for folder in folders:
    folder_path = nifty_pdf_data_path / folder
    if not folder_path.is_dir():
        continue

    month = folder_path.stem[:3]
    year = folder_path.stem[3:]
    date = pd.to_datetime(f"{month} {year}", format="%b %Y")

    for file in glob(str(folder_path / "*.csv")):
        filename = Path(file)
        if "large" in filename.stem.lower() and "0" in filename.stem.lower():
            print(f"filename: {filename}")
            shutil.copy(filename, output_dir / f"{date.strftime('%Y-%m-%d')}.csv")

    # break

In [5]:
pdf_data_path = Path("pdf data")
pdf_data_folder = os.listdir(pdf_data_path)

In [6]:
rebalance_df =pd.DataFrame()
old = pdf_data_folder[0]
old_csv = pd.read_csv(pdf_data_path / old)
old_csv = old_csv[["Security Name", "Symbol"]]
for filename in pdf_data_folder[1:]:
    file_path = pdf_data_path / filename
    new_csv = pd.read_csv(file_path)
    new_csv = new_csv[["Symbol", "Security Name"]]
    left_unique_rows = pd.merge(old_csv, new_csv, on=["Symbol", "Security Name"], how="left", indicator=True)
    left_unique_rows = left_unique_rows[left_unique_rows["_merge"] == "left_only"].drop(columns=["_merge"])
    left_unique_rows["action"] = "excluded"
    left_unique_rows["Index"] = "NIFTY MidSmallcap 400"
    left_unique_rows["date"] = pd.to_datetime(filename.split(".")[0])
    left_unique_rows = left_unique_rows[["date", "Index", "action","Security Name", "Symbol"]]
    left_unique_rows.rename(columns={"Security Name":"company"}, inplace=True)
    # display(left_unique_rows)
    
    right_unique_rows = pd.merge(old_csv, new_csv, on=["Symbol", "Security Name"], how="right", indicator=True)
    right_unique_rows = right_unique_rows[right_unique_rows["_merge"] == "right_only"].drop(columns=["_merge"])
    right_unique_rows["date"] = pd.to_datetime(filename.split(".")[0])
    right_unique_rows["action"] = "included"
    right_unique_rows["Index"] = "NIFTY MidSmallcap 400"
    right_unique_rows = right_unique_rows[["date", "Index", "action","Security Name", "Symbol"]]
    right_unique_rows.rename(columns={"Security Name":"company"}, inplace=True)
    # display(right_unique_rows)
    
    rebalance_df = pd.concat([rebalance_df, left_unique_rows, right_unique_rows], ignore_index=True)
    # display(rebalance_df)
    old_csv = new_csv
    # break

In [7]:
rebalance_df

,date,Index,action,company,Symbol
0,2017-08-01,NIFTY MidSmallcap 400,excluded,Fag Bearings India Ltd.,FAGBEARING
1,2017-08-01,NIFTY MidSmallcap 400,included,Schaeffler India Ltd.,SCHAEFFLER
2,2017-09-01,NIFTY MidSmallcap 400,excluded,Amtek Auto Ltd.,AMTEKAUTO
3,2017-09-01,NIFTY MidSmallcap 400,excluded,Anant Raj Ltd.,ANANTRAJ
4,2017-09-01,NIFTY MidSmallcap 400,excluded,Asahi India Glass Ltd.,ASAHIINDIA
...,...,...,...,...,...
697,2022-03-01,NIFTY MidSmallcap 400,included,Shyam Metalics and Energy Ltd.,SHYAMMETL
698,2022-03-01,NIFTY MidSmallcap 400,included,Star Health and Allied Insurance Company Ltd.,STARHEALTH
699,2022-03-01,NIFTY MidSmallcap 400,included,Tata Investment Corporation Ltd.,TATAINVEST
700,2022-03-01,NIFTY MidSmallcap 400,included,Vijaya Diagnostic Centre Ltd.,VIJAYA


In [8]:
rebalance_df.to_csv("rebalance_data.csv", index=False)